# Main 72건 적격성 감사 재현

2026-09-08. 이 노트북은 원문 위치·해시·집계·중복 임계값을 재확인한다. 영역/문맥 수동 판정 자체를 자동 검증하거나 모델 응답을 재채점하지 않는다. `pypdf`가 설치된 Python 환경에서 프로젝트 또는 이 노트북의 디렉터리를 작업 디렉터리로 사용한다.


In [1]:
from pathlib import Path
import sys, json
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'data/main_cases.csv').exists())
sys.path.insert(0, str(root/'tools'))
from audit_main72_eligibility_20260908 import collect
result = collect()
assert result['all_mechanical_checks_pass'], result['checks']
print(json.dumps(result['checks'], ensure_ascii=False, indent=2))


{
  "72_unique_cases": true,
  "72_unique_candidates": true,
  "72_explicit_decisions": true,
  "modal_balance": true,
  "source_balance": true,
  "frozen_tier_counts": true,
  "all_prior_frozen_hashes_unchanged": true,
  "pdf_hash_3GPP TS 23.501": true,
  "pdf_hash_3GPP TS 24.501": true,
  "all_source_text_located": true,
  "all_case_source_hashes_match": true,
  "clause_findings_accounted_for": true,
  "no_main_same_modal_jaccard_060": true,
  "no_pilot_jaccard_072": true,
  "modal_token_present": true
}


In [2]:
print(json.dumps(result['decision_counts'], ensure_ascii=False, indent=2))
print(json.dumps(result['by_modal'], ensure_ascii=False, indent=2))
print(json.dumps(result['clause_mismatches'], ensure_ascii=False, indent=2))


{
  "ELIGIBLE": 50,
  "INELIGIBLE": 10,
  "HOLD": 11,
  "ELIGIBLE_WITH_RECORDED_EXCEPTION": 1
}
{
  "can": {
    "ELIGIBLE": 9,
    "HOLD": 2,
    "INELIGIBLE": 1
  },
  "may": {
    "ELIGIBLE": 10,
    "INELIGIBLE": 1,
    "HOLD": 1
  },
  "shall": {
    "ELIGIBLE": 8,
    "INELIGIBLE": 1,
    "HOLD": 3
  },
  "shall_not": {
    "ELIGIBLE": 9,
    "INELIGIBLE": 2,
    "HOLD": 1
  },
  "should": {
    "ELIGIBLE": 8,
    "INELIGIBLE": 3,
    "HOLD": 1
  },
  "should_not": {
    "INELIGIBLE": 2,
    "HOLD": 3,
    "ELIGIBLE": 6,
    "ELIGIBLE_WITH_RECORDED_EXCEPTION": 1
  }
}
[
  {
    "case_id": "MAIN-CAN-005",
    "source_match": true,
    "declared_clause": "4.2.8.1",
    "source_clause": "4.2.8.1A",
    "case_hash_matches": true
  }
]


## 수동 판정의 추적과 한계

`decisions.tsv`는 사례별 명시적 판정 입력이다. 아래 검사는 전수 기록·집계의 일치만 검사한다. 문맥 보류는 확정 부적격과 다르며, 50/51개 잔여 표본을 새 동결 데이터셋으로 승격하지 않는다.


In [3]:
from collections import Counter
audit_dir = root/'outputs/analysis/main72_eligibility_20260908'
audit = json.loads((audit_dir/'eligibility_audit.json').read_text())
assert len(audit['rows']) == len({r['case_id'] for r in audit['rows']}) == 72
assert dict(Counter(r['status'] for r in audit['rows'])) == result['decision_counts']
for r in audit['rows']:
    if r['status'] != 'ELIGIBLE':
        print(r['case_id'], r['status'], r['rationale_ko'])


MAIN-SH-003 INELIGIBLE PDF 100쪽의 두 번째 access가 다른 PLMN인 경우라는 상위 bullet 안이다. 그 조건은 원문 입력과 질문에서 빠졌다.
MAIN-SH-004 HOLD PILOT-SH-001과 5GC CP NF의 service-based interaction이라는 핵심 actor/action이 겹친다. only와 where identified 차이가 있어 동일 표본으로 단정하지 않고 의미 중복 여부를 보류한다.
MAIN-SH-007 HOLD then은 같은 문단의 single-registration·N26 지원·S1→N1·5GMM-CONNECTED mapped context 생성 과정을 잇는다. 현 입력/질문은 그 적용 조건을 충분히 담지 않는다.
MAIN-SH-011 HOLD PDF 56쪽 첫 mapped-context 생성 문단의 후속 문장이다. newly derived K'_AMF의 도출 조건인 single-registration·N26·S1→N1·5GMM-CONNECTED 범위 보충이 필요하다.
MAIN-MAY-005 INELIGIBLE PDF 76쪽 4.3.1 NOTE 2 내부 문장이다. 모델 실행 전 NOTE 제외 규약을 위반한다.
MAIN-MAY-010 HOLD 같은 문단은 SERVICE REQUEST 또는 CONTROL PLANE SERVICE REQUEST의 integrity failure 처리다. 발췌는 emergency PDU session과 integrity failure만 남겨 대상 메시지 범위를 잃었다.
MAIN-CAN-003 HOLD The architecture는 4.4.8.4의 IETF DetNet 통합 구조를 가리킨다. 발췌와 질문의 일반적인 5GS architecture만으로 그 범위를 확정하기 어렵다.
MAIN-CAN-004 INELIGIBLE PDF 51쪽 4.2.5 NOTE 2 내부다. only authorized NFs 문장이라는 의미와 별개로 사전 NOTE 제외 규칙 위반이다.
M